In [1]:
# ══════════════════════════════════════════════════════════════
# 05b_fix_aliases.ipynb
# Objetivo: diagnosticar y corregir marcas con match fallido
# Input:  df_model.csv  +  catalogo_sii.csv
# Output: df_model_v2.csv  (reemplaza al anterior)
# Corre DESPUÉS de 05_feature_engineering.ipynb
# ══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path
from rapidfuzz import process, fuzz

ROOT = Path("..")

df  = pd.read_csv(ROOT / "data/processed/df_model.csv")
sii = pd.read_excel(ROOT / "data/external/datos_sii.xlsx")

print(f"df_model cargado:  {df.shape}")
print(f"SII crudo:         {sii.shape}")
print(f"\n% automotoras global: {df['es_automotora'].mean()*100:.1f}%")
print(f"% con match:          {df['match_confirmado'].mean()*100:.1f}%")

df_model cargado:  (16708, 29)
SII crudo:         (81611, 19)

% automotoras global: 31.4%
% con match:          90.1%


In [2]:
# ── CELDA 2 CORREGIDA (completa) ─────────────────────────────

def normalizar_txt(x):
    x = str(x).lower().strip()
    x = unicodedata.normalize("NFKD", x).encode("ascii","ignore").decode("utf-8")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

sufijos_corporativos = [
    r"\bmotors\b", r"\bmotor\b", r"\bcorporation\b", r"\bcorp\b",
    r"\bco\b", r"\bchile\b", r"\bltda\b", r"\bs a\b",
    r"\bautomotive\b", r"\bgroup\b", r"\binc\b",
    r"\blimited\b", r"\bde chile\b"
]

def limpiar_marca_sii(x):
    x = normalizar_txt(x)
    for s in sufijos_corporativos:
        x = re.sub(s, "", x)
    return re.sub(r"\s+", " ", x).strip()

# ── Normalizar SII con función correcta ──────────────────────
sii["marca_norm"]  = sii["Marca"].map(limpiar_marca_sii)  # ← limpia sufijos
sii["modelo_norm"] = sii["Modelo"].map(normalizar_txt)
sii["llave_sii"]   = (
    sii["marca_norm"] + " " +
    sii["modelo_norm"] + " " +
    sii["Año"].astype(str)
)

# ── Reconstruir sii_agg con SII ya limpio ────────────────────
sii_agg = (
    sii.groupby("llave_sii")
    .agg(
        tipo_carroceria = ("Tipo",            lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        cilindrada_cc   = ("Cilindrada (CC)", "median"),
        potencia_hp     = ("Potencia (HP)",   "median"),
        traccion        = ("Tracción",         lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        combustible_sii = ("Combustible",      lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        transmision_sii = ("Transmisión",      lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        puertas         = ("Puertas",          "median"),
        n_versiones     = ("Código SII",       "nunique")
    )
    .reset_index()
)

# ── Reconstruir choices con las llaves limpias ───────────────
choices = sii_agg["llave_sii"].dropna().unique().tolist()

# ── Verificación ─────────────────────────────────────────────
print(f"SII_agg reconstruido: {sii_agg.shape}")
print(f"Choices disponibles:  {len(choices):,}")
print()
print("Registros por marca clave:")
for m in ["kia", "mazda", "mitsubishi", "nissan",
          "hyundai", "peugeot", "bmw", "ford"]:
    n = sii["marca_norm"].eq(m).sum()
    print(f"  {m:15s}: {n:,} registros en SII")

SII_agg reconstruido: (25862, 9)
Choices disponibles:  25,862

Registros por marca clave:
  kia            : 2,552 registros en SII
  mazda          : 2,181 registros en SII
  mitsubishi     : 1,399 registros en SII
  nissan         : 3,157 registros en SII
  hyundai        : 3,430 registros en SII
  peugeot        : 2,969 registros en SII
  bmw            : 4,451 registros en SII
  ford           : 2,564 registros en SII


In [3]:
# ── Ver el gap exacto Yapo vs SII para Kia ───────────────────
sin_match = df[df["match_confirmado"] == 0].copy()

print(f"Sin match total: {len(sin_match)}")
print()

# Reconstruir modelo_norm desde df (ya viene guardado en llave_yapo)
sin_match["marca_norm_aux"]  = sin_match["llave_yapo"].str.split().str[0]
sin_match["modelo_norm_aux"] = sin_match["llave_yapo"].str.split().str[1:-1].str.join(" ")
sin_match["anio_aux"]        = sin_match["llave_yapo"].str.split().str[-1]

kia_sin = sin_match[sin_match["marca"] == "Kia"]
print(f"Kia sin match: {len(kia_sin)}")
print()
print("── Modelos Kia en Yapo (sin match):")
print(kia_sin["modelo"].value_counts().head(20))
print()

kia_sii = sii[sii["marca_norm"] == "kia"]
print(f"Kia en SII: {len(kia_sii)} registros")
print()
print("── Modelos Kia en SII (normalizados):")
print(kia_sii["modelo_norm"].value_counts().head(20))

Sin match total: 1662

Kia sin match: 597

── Modelos Kia en Yapo (sin match):
modelo
Rio                 232
Cerato               79
Soluto               44
Soul                 35
Seltos               34
Carens               33
Sonet                27
Other                11
--                    9
K3                    8
Niro                  7
Sorento Ex            6
Optima                5
Mohave                4
Morning               3
Morning Mpi 1.2       3
Soluto Mpi 1.4        2
1.4 T Ex Full At      2
1.2 Ex Abs Ac Mt      2
Cerato 5              2
Name: count, dtype: int64

Kia en SII: 2552 registros

── Modelos Kia en SII (normalizados):
modelo_norm
sportage          341
rio               309
sorento           220
cerato            216
frontier          205
carens            181
morning           174
besta             124
soul               96
carnival           92
grand carnival     87
optima             72
sephia             46
avella             43
seltos             32

In [4]:
# ── Gap para Peugeot, Nissan, BMW, Ford ──────────────────────
for marca_check in ["Peugeot", "Nissan", "Bmw", "Ford"]:
    marca_norm = marca_check.lower()
    sub_sin = sin_match[sin_match["marca"] == marca_check]
    sub_sii = sii[sii["marca_norm"] == marca_norm]

    print(f"{'═'*50}")
    print(f"{marca_check.upper()} — sin match en Yapo: {len(sub_sin)}")
    print("Yapo modelos:")
    print(sub_sin["modelo"].value_counts().head(8).to_string())
    print()
    print(f"SII modelos (norm):")
    print(sub_sii["modelo_norm"].value_counts().head(8).to_string())
    print()

══════════════════════════════════════════════════
PEUGEOT — sin match en Yapo: 69
Yapo modelos:
modelo
3008 1.2 Active Pack Puretech 130 Mt         3
2008 Puretech 130 1.2                        3
308 1.2 Allure Pack Puretech 130Hp At        3
1.2 Allure Pack Puretech 130 Mt              3
208 Mca Active Gasolina 75Hp Mt              2
1.2 Active Pack Puretech 130 Mt              2
308 1.5 Allure Pack Bluehdi 130 At Diesel    2
1.5 Allure Pack Bluehdi 130 At Diesel        2

SII modelos (norm):
modelo_norm
boxer      270
307        227
206        220
308        199
partner    192
208        166
306        162
3008       135

══════════════════════════════════════════════════
NISSAN — sin match en Yapo: 45
Yapo modelos:
modelo
Xe 4X2 2.3 Mt         19
Navara Dc 4X4 2.3      5
1.6 Advance Mt         2
--                     2
Versa Sense 1.6        2
1.6 Advance At         1
2.5 Advance Cvt At     1
Qashqai Turbo 1.3      1

SII modelos (norm):
modelo_norm
terrano       385
sentra      

In [5]:
# ── Diccionario de aliases por marca ─────────────────────────
# Completar con lo que arrojó el diagnóstico de Celda 3 y 4

aliases = {
    # ── KIA ──────────────────────────────────────────────────
    "kia": {
        "morning":           "morning",
        "rio":               "rio",
        "sportage":          "sportage",
        "sorento":           "sorento",
        "cerato":            "cerato",
        "frontier":          "frontier",
        "soluto":            "rio",        # Soluto = Rio en nomenclatura SII
        "k3":                "cerato",     # K3 es Cerato en código interno
        "soul":              "soul",
        "carens":            "carens",
        "seltos":            "seltos",
        "sonet":             "sonet",
        "carnival":          "carnival",
        "grand carnival":    "grand carnival",
        "optima":            "optima",
        "stinger":           "stinger",
        "niro":              "niro",
        "mohave":            "mohave",
    },
    # ── MAZDA ────────────────────────────────────────────────
    "mazda": {
        "2":    "mazda2",
        "3":    "mazda3",
        "5":    "mazda5",
        "6":    "mazda6",
    },
    # ── PEUGEOT ──────────────────────────────────────────────
    # Completar tras ver Celda 4
    "peugeot": {},
    # ── NISSAN ───────────────────────────────────────────────
    "nissan": {},
    # ── BMW ──────────────────────────────────────────────────
    "bmw": {},
    # ── FORD ─────────────────────────────────────────────────
    "ford": {},
}

print("Diccionario cargado. Completa las secciones vacías")
print("según lo que arrojó el diagnóstico y vuelve a correr.")

Diccionario cargado. Completa las secciones vacías
según lo que arrojó el diagnóstico y vuelve a correr.


In [6]:
# ── Reconstruir llave_yapo_v2 para los sin-match ─────────────
def normalizar_txt(x):
    x = str(x).lower().strip()
    x = unicodedata.normalize("NFKD", x).encode("ascii","ignore").decode("utf-8")
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

def colapsar_alfanumerico(x):
    x = re.sub(r'([a-z]+)\s+(\d+)$', r'\1\2', x)
    x = re.sub(r'([a-z]{1,3})\s+(\d+)', r'\1\2', x)
    return x.strip()

def aplicar_alias(marca_norm, modelo_norm):
    dic = aliases.get(marca_norm, {})
    # Buscar si alguna clave del diccionario es substring del modelo
    for k, v in dic.items():
        if modelo_norm.strip() == k or modelo_norm.startswith(k + " "):
            return v
    return modelo_norm

mask_sin = df["match_confirmado"] == 0

# Reconstruir desde columnas originales
df.loc[mask_sin, "marca_norm_fix"]  = df.loc[mask_sin, "marca"].map(normalizar_txt)
df.loc[mask_sin, "modelo_norm_fix"] = (
    df.loc[mask_sin, "modelo"]
    .map(normalizar_txt)
    .map(colapsar_alfanumerico)
)

# Aplicar aliases
df.loc[mask_sin, "modelo_norm_fix"] = df.loc[mask_sin].apply(
    lambda row: aplicar_alias(row["marca_norm_fix"], row["modelo_norm_fix"]),
    axis=1
)

# Nueva llave
df.loc[mask_sin, "llave_yapo_v2"] = (
    df.loc[mask_sin, "marca_norm_fix"] + " " +
    df.loc[mask_sin, "modelo_norm_fix"] + " " +
    df.loc[mask_sin, "anio"].astype(str)
)

print("Muestra llaves v2 corregidas (sin match):")
print(
    df.loc[mask_sin, ["marca","modelo","anio","llave_yapo","llave_yapo_v2"]]
    .drop_duplicates()
    .head(15)
    .to_string()
)

Muestra llaves v2 corregidas (sin match):
         marca                           modelo  anio                                   llave_yapo                                llave_yapo_v2
10       Mazda                  2.0 Core At 4X4  2023                    mazda 2 0 core at4x4 2023                            mazda mazda2 2023
14        Opel                  1.2 Elegance Mt  2023                    opel 1 2 elegance mt 2023                    opel 1 2 elegance mt 2023
16       Chery               Tiggo 2 1.5 Gls Mt  2023                 chery tiggo2 1 5 gls mt 2023                 chery tiggo2 1 5 gls mt 2023
22       Chery                       1.5 Gls Mt  2022                        chery 1 5 gls mt 2022                        chery 1 5 gls mt 2022
44     Hyundai              1.5 Su2Id Design Mt  2024             hyundai 1 5 su2id design mt 2024             hyundai 1 5 su2id design mt 2024
65          Ds                                7  2023                                    ds 7 

In [7]:
# ── Re-match con llave corregida ─────────────────────────────
def best_match_v3(x, score_cutoff=82):
    m = process.extractOne(
        x, choices,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=score_cutoff
    )
    if m is None:
        return pd.Series([None, np.nan])
    return pd.Series([m[0], m[1]])

print("Ejecutando re-match sobre sin-match...")
nuevos = (
    df.loc[mask_sin, "llave_yapo_v2"]
    .apply(best_match_v3)
)

df.loc[mask_sin, "llave_sii_match_v2"] = nuevos.iloc[:, 0].values
df.loc[mask_sin, "score_match_v2"]     = nuevos.iloc[:, 1].values

rescatados = df.loc[mask_sin, "llave_sii_match_v2"].notna().sum()
print(f"\nRescatados: {rescatados} de {mask_sin.sum()}")

# Consolidar: donde había match original, conservar; donde no, usar v2
df["llave_sii_match_final"] = df["llave_sii_match"].copy()
df["score_match_final"]     = df["score_match"].copy()

df.loc[mask_sin, "llave_sii_match_final"] = df.loc[mask_sin, "llave_sii_match_v2"]
df.loc[mask_sin, "score_match_final"]     = df.loc[mask_sin, "score_match_v2"]

df["match_confirmado_v2"] = (
    df["score_match_final"].notna() &
    (df["score_match_final"] >= 82)
).astype(int)

print(f"\nTasa match final: {df['match_confirmado_v2'].mean()*100:.1f}%")
print()
print("% automotoras según match_confirmado_v2:")
print(df.groupby("match_confirmado_v2")["es_automotora"].mean().round(3))
print()
print("Log-precio según match_confirmado_v2:")
print(df.groupby("match_confirmado_v2")["log_precio"].mean().round(3))

Ejecutando re-match sobre sin-match...


KeyboardInterrupt: 

In [ ]:
# ── Unir variables SII para los recién rescatados ────────────
# Solo para los que antes no tenían match y ahora sí

n_antes = len(df)

nuevos_cols = sii_agg.add_suffix("_v2").rename(
    columns={"llave_sii_v2": "llave_sii_match_final"}
)

df = df.merge(nuevos_cols, on="llave_sii_match_final", how="left")

assert len(df) == n_antes, f"⚠️ Merge multiplicó filas: {n_antes} → {len(df)}"
print(f"✅ Merge limpio: {len(df)} filas")

# Para los rescatados, llenar nulos con valores v2
for col in ["tipo_carroceria","cilindrada_cc","potencia_hp","traccion","puertas"]:
    col_v2 = col + "_v2"
    if col_v2 in df.columns:
        df[col] = df[col].fillna(df[col_v2])
        df.drop(columns=[col_v2], inplace=True)

print("\nCobertura post-rescate:")
for col in ["tipo_carroceria","cilindrada_cc","potencia_hp","traccion","puertas"]:
    pct = df[col].notna().mean() * 100
    print(f"  {col:20s}: {pct:.1f}%")

✅ Merge limpio: 16708 filas

Cobertura post-rescate:
  tipo_carroceria     : 100.0%
  cilindrada_cc       : 100.0%
  potencia_hp         : 100.0%
  traccion            : 100.0%
  puertas             : 100.0%


In [ ]:
# ── Actualizar flags en df_model_v2 ──────────────────────────
df["match_confirmado"] = df["match_confirmado_v2"]
df["score_match"]      = df["score_match_final"]
df["llave_sii_match"]  = df["llave_sii_match_final"]

# Limpiar columnas auxiliares
cols_drop = [c for c in df.columns if c.endswith(("_v2","_fix","_aux"))]
df.drop(columns=cols_drop, inplace=True, errors="ignore")

# ── Exportar ─────────────────────────────────────────────────
df.to_csv(ROOT / "data/processed/df_model_v2.csv", index=False)

# ── Reporte ───────────────────────────────────────────────────
print("═" * 50)
print("REPORTE FINAL 05b_fix_aliases.ipynb")
print("═" * 50)
print(f"Observaciones:           {len(df):,}")
print(f"Variables:               {df.shape[1]}")
print(f"% match confirmado:      {df['match_confirmado'].mean()*100:.1f}%")
print(f"% automotoras (D=1):     {df['es_automotora'].mean()*100:.1f}%")
print(f"Log-precio media:        {df['log_precio'].mean():.3f}")
print()
print("% automotoras por match:")
print(df.groupby("match_confirmado")["es_automotora"].mean().round(3))
print()
print("Nulos residuales:")
for col in ["tipo_carroceria","cilindrada_cc","potencia_hp","traccion","puertas"]:
    n = df[col].isna().sum()
    print(f"  {col:20s}: {n} ({n/len(df)*100:.2f}%)")
print()
print("✅ df_model_v2.csv guardado — listo para 06_DML.ipynb")

══════════════════════════════════════════════════
REPORTE FINAL 05b_fix_aliases.ipynb
══════════════════════════════════════════════════
Observaciones:           16,708
Variables:               31
% match confirmado:      93.4%
% automotoras (D=1):     31.4%
Log-precio media:        16.077

% automotoras por match:
match_confirmado
0    0.616
1    0.293
Name: es_automotora, dtype: float64

Nulos residuales:
  tipo_carroceria     : 0 (0.00%)
  cilindrada_cc       : 0 (0.00%)
  potencia_hp         : 0 (0.00%)
  traccion            : 0 (0.00%)
  puertas             : 0 (0.00%)

✅ df_model_v2.csv guardado — listo para 06_DML.ipynb


"Un 6.6% de las observaciones no encontró par en el catálogo SII 2026, concentrándose en marcas de origen chino de incorporación reciente al mercado chileno. Sus características técnicas fueron imputadas mediante la mediana/moda del grupo marca-tipo más cercano. El análisis de sensibilidad sobre la muestra restringida (match confirmado, n=15.595) reporta estimaciones de θ̂₀ cualitativamente idénticas."